In [ ]:
import numpy as np
import pandas as pd

import os.path as op
import numpy as np

from tqdm.contrib.itertools import product
import pingouin
from tms_risk.utils.data import get_subjects, get_all_behavior, get_pdf, get_decoding_info, get_all_apriori_roi_labels, Subject
import seaborn as sns

import matplotlib.pyplot as plt


stimulation_palette = sns.color_palette()[2:4]
stimulation_order = ['Vertex', 'IPS']

bids_folder = '/data/ds-tmsrisk/'

# Only TMS target info

In [ ]:
subjects = [int(sub.subject) for sub in get_subjects(all_tms_conditions=True, exclude_outliers=True)]
sessions = [1, 2, 3]
pca_confounds = [False]
denoise = [True]
smoothed = [False]
# masks = ['NPCr1cm-surface', 'NPCr1cm-cluster', 'NPCr2cm-surface', 'NPCr2cm-cluster']
# n_voxels = [0, 1, 100]
masks = ['NPCr2cm-cluster']
n_voxels = [1]
natural_space = [True]


pred = []
pdfs = []
for (sub, session, pcac, den, smooth, mask, nv, ns) in product(subjects, sessions, pca_confounds, denoise, smoothed, masks, n_voxels, natural_space):

    if not ((session == 1) & (nv == 0)):
        pdfs.append(get_pdf(sub, session, pcac, den, smooth, '/data/ds-tmsrisk/', mask, nv, ns, new_parameterisation=True))
        pred.append(get_decoding_info(sub, session, pcac, den, smooth, '/data/ds-tmsrisk', mask, nv, ns))

In [ ]:
pred = pd.concat(pred)
pdf = pd.concat(pdfs)
df = get_all_behavior(drop_no_responses=False)
pred = pred.join(df, how='inner')
r1 = pred.groupby(['subject', 'session', 'stimulation_condition', 'pca', 'glm', 'smoothed', 'mask', 'n_voxels']).apply(lambda d: pingouin.corr(d['E'], d['n1']))
r2_ = pred.groupby(['subject', 'session', 'stimulation_condition', 'pca', 'glm', 'smoothed', 'mask', 'n_voxels', 'run']).apply(lambda d: pingouin.corr(d['E'], d['n1']))
r2 = r2_.groupby(['subject', 'session', 'stimulation_condition', 'pca', 'glm', 'smoothed', 'mask', 'n_voxels'])[['r']].mean(numeric_only=True)

In [ ]:
r2.groupby(['n_voxels', 'stimulation_condition', 'mask']).size()

In [ ]:
pred['Order'] = pred['risky_first'].map({True:'Risky first', False:'Safe first'})

In [ ]:
r3_ = pred.groupby(['subject', 'session', 'stimulation_condition', 'pca', 'glm', 'smoothed', 'mask', 'n_voxels', 'run', 'Order']).apply(lambda d: pingouin.corr(d['E'], d['n1']))
r3 = r3_.groupby(['subject', 'session', 'stimulation_condition', 'pca', 'glm', 'smoothed', 'mask', 'n_voxels', 'Order'])['r'].mean(numeric_only=True)

In [ ]:
pred['Stimulation condition'] = pred.index.get_level_values('stimulation_condition').map({'vertex': 'Vertex', 'ips': 'IPS', 'baseline':'Baseline'})
pred.set_index('Stimulation condition', append=True, inplace=True)

In [ ]:
g = sns.catplot(x='n_voxels', y='r', hue='stimulation_condition',data=r2.reset_index(), col='mask', kind='swarm', row='smoothed', dodge=True, ci=67)
g.map(lambda *args, **kwargs: plt.axhline(0, c='k', ls='--'))

In [ ]:
sns.set(font_scale=1.6, style='white', font='Helvetica')

g = sns.catplot(x='stimulation_condition', y='r', data=r2.drop('baseline', level='stimulation_condition').reset_index(), col='n_voxels', kind='point', dodge=True, errorbar='se', palette=['k'], height=6.)
g.map(lambda *args, **kwargs: plt.axhline(0, c='k', ls='--'))


g.set(ylabel='Decoding accuracy (r)', xlabel='Stimulation condition')
g.set_xticklabels(['Vertex', 'IPS'])
g.set(yticks=[0.0, 0.05, 0.1, 0.15])
g.set_titles('')

g.savefig(op.join(bids_folder, 'derivatives', 'figures', 'decoding.pdf'))

In [ ]:
r2.groupby(['n_voxels', 'stimulation_condition', 'mask']).mean(numeric_only=True)

In [ ]:
tmp = r2.drop('baseline', level='stimulation_condition').reset_index()

tmp.groupby(['n_voxels']).apply(lambda d: pingouin.rm_anova(d, 'r', 'stimulation_condition', 'subject'))
# pingouin.rm_anova(tmp, 'r', 'stimulation_condition', 'subject')

In [ ]:
g = sns.catplot(hue='stimulation_condition', y='r', x='Order', data=r3.drop('baseline', level='stimulation_condition').reset_index(), col='mask', kind='point', dodge=True, ci=67, palette=stimulation_palette)
g.map(lambda *args, **kwargs: plt.axhline(0, c='k', ls='--'))


g.set(ylabel='Decoding accuracy (r)')




g.savefig(op.join(bids_folder, 'derivatives', 'figures', 'decoding.pdf'))

In [ ]:
tmp = r3.drop('baseline', level='stimulation_condition').reset_index()

pingouin.rm_anova(tmp, 'r', ['stimulation_condition', 'Order'], 'subject')

# Get the noise increase for each subject for flexible model

In [ ]:
from tms_risk.behavior.fit_model import build_model, get_data

In [ ]:
tmp = r2.xs('NPCr2cm-cluster', 0, 'mask').xs(1, 0, 'n_voxels').droplevel('session').unstack('stimulation_condition')['r']

pingouin.ttest(tmp['ips'], tmp['vertex'], True)

In [ ]:
pred['n_bin'] = pd.cut(pred['n1'], np.arange(5, 100, 5), labels=np.arange(7.5, 97.5, 5))

In [ ]:
pred['n_bin_q'] = pd.qcut(pred['n1'], 7)

pred['n_bin_q'] = pred['n_bin_q'].apply(lambda x: x.mid)

In [ ]:
tmp = pred.xs('NPCr2cm-cluster', 0, 'mask').xs(1, 0, 'n_voxels').groupby(['subject', 'stimulation_condition', 'n_bin']).mean(numeric_only=True).drop('baseline', level='stimulation_condition')

sns.lineplot(x='n_bin', y='sd', hue='stimulation_condition', data=tmp.reset_index(), errorbar='se')

In [ ]:
tmp = pred.xs('NPCr2cm-cluster', 0, 'mask').xs(1, 0, 'n_voxels').groupby(['subject', 'Stimulation condition', 'n_bin_q']).mean(numeric_only=True).drop('Baseline', level='Stimulation condition')

g = sns.lineplot(x='n_bin_q', y='sd', hue='Stimulation condition', data=tmp.reset_index(), errorbar='se', palette=stimulation_palette, hue_order=stimulation_order, legend=False)

g.set(xlabel='Mu (binned)', ylabel='Decoded uncertainty (sd of posterior)')



# g.add_legend()
sns.despine()

plt.savefig(op.join(bids_folder, 'derivatives', 'figures', 'decoding_sd.pdf'))
# g.savefig()

In [ ]:
pred['error'] = pred['n1'] - pred['E']
pred['abs(error)'] = np.abs(pred['error'])

In [ ]:
tmp = pred.xs('NPCr2cm-cluster', 0, 'mask').xs(1, 0, 'n_voxels').groupby(['subject', 'Stimulation condition', 'n_bin_q']).mean(numeric_only=True).drop('Baseline', level='Stimulation condition')

sns.lineplot(x='n_bin_q', y='abs(error)', hue='Stimulation condition', data=tmp.reset_index(), errorbar='se', palette=stimulation_palette, hue_order=stimulation_order, legend=False)

plt.xlim(5, 40)

# 12 a-priori ROIs

In [ ]:
subjects = [int(sub.subject) for sub in get_subjects(all_tms_conditions=True, exclude_outliers=True)]
sessions = [1, 2, 3]
pca_confounds = [False]
denoise = [True]
smoothed = [False]
# masks = ['NPCr1cm-surface', 'NPCr1cm-cluster', 'NPCr2cm-surface', 'NPCr2cm-cluster']
# n_voxels = [0, 1, 100]
masks = get_all_apriori_roi_labels()
n_voxels = [1]
natural_space = [True]


pred = []
pdfs = []
for (sub, session, pcac, den, smooth, mask, nv, ns) in product(subjects, sessions, pca_confounds, denoise, smoothed, masks, n_voxels, natural_space):

    if not ((session == 1) & (nv == 0)):
        pdfs.append(get_pdf(sub, session, pcac, den, smooth, '/data/ds-tmsrisk/', mask, nv, ns, new_parameterisation=True))
        pred.append(get_decoding_info(sub, session, pcac, den, smooth, '/data/ds-tmsrisk', mask, nv, ns))

In [ ]:
# This cell is a duplicate of the data-loading block above. When the
# notebook is run linearly it would error (pred/pdfs are already
# concat'd DataFrames at this point). Left as a no-op so the
# exploratory cells below can re-use the `r2` computed at the top.


In [ ]:
sns.catplot(x='mask', y='r', hue='stimulation_condition', data=r2.reset_index(), kind='bar', aspect=3.)

In [ ]:
r2.droplevel('session').drop('baseline', level='stimulation_condition')['r'].unstack('stimulation_condition').groupby(['mask']).apply(lambda d: pingouin.ttest(d['ips'], d['vertex'], paired=True)).sort_values('p-val')

In [ ]:
tmp = r2.droplevel('session').unstack(['stimulation_condition'])['r']

diff = (tmp['ips'] - tmp['vertex']).to_frame('diff')

sns.catplot(x='mask', y='diff', data=diff.reset_index(), kind='point', aspect=2.5)
plt.axhline(0.0, c='k', ls='--')

In [ ]:
sns.heatmap(r2['r'].unstack('mask').corr(), cmap='viridis')

In [ ]:
sns.catplot(x='mask', y='r', data=r2.groupby(['subject', 'mask']).mean(numeric_only=True).reset_index(), kind='bar', aspect=3., color='gray', errorbar='se')

In [ ]:
r2.groupby(['subject', 'session', 'mask']).size().groupby(['mask', 'session']).size()